# Cierre mensual

Lo publicado (enero-setiembre 2026 y cada mes ya cargado) **no cambia**: cada cierre publica solo su mes, más el G04 de Impulso mientras es provisional. Guía: `README.md`.

**Probar** (cuantas veces quieras, no deja rastro): `MODO = 'dry_run'` → corre todo → revisa la sección 3 y el Excel de la vista previa (sección 4).
**Cerrar el mes** (cuando las ventas del mes estén cerradas): `dry_run` → revisar → `MODO = 'cargar'` → exportar (sección 5).

In [ ]:
# ── PARÁMETROS (lo único que se edita) ──
MES_CIERRE = '2026-10-01'                    # día 1 del mes que se cierra: '2026-10-01', '2026-11-01', '2026-12-01', ...
PROGRAMAS  = ['cpb', 'cpm', 'champion_motor', 'surtido_motor']
MODO       = 'dry_run'                       # 'dry_run' = prueba (no escribe nada, no deja estado) | 'cargar' = publica

# Opcionales: dejar así salvo decisión explícita (todos quedan registrados)
VALIDAR_CERRADOS = False                     # CPB/CPM: True recalcula también los meses cerrados para medir su deriva (más lento)
TOLERANCIA_CELDA = None                      # None = 0.25 pp en brechas y 1 % (y 1 kg) en toneladas, por celda mes x categoría
ACEPTAR_DERIVA, MOTIVO_DERIVA = False, None  # continuar una carga bloqueada por una deriva explicada
REPROCESO, REABRIR_DESDE, MOTIVO = False, None, None   # corregir un cierre publicado / reabrir meses cerrados (exigen MOTIVO)

In [ ]:
import sys, importlib
from pathlib import Path

RAIZ = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(RAIZ / 'src'))
for m in [m for m in sys.modules if m == 'mediciones' or m.startswith('mediciones.')]:
    del sys.modules[m]
importlib.invalidate_caches()

import pandas as pd
import mediciones
from mediciones.runner import correr_mes
from mediciones.salidas import exportar, vista_previa
print('carpeta de trabajo:', RAIZ)
print('código en uso     :', mediciones.__file__)

## 1. Archivos que tienen que estar
Todo `OK`. Si algo dice `FALTA`, ver `README.md` (sección *Qué colocar*).

In [ ]:
SD, CD = 'Surtido Ideal - Medición', 'Champion Minimayorista - Medición'
MESES_ES = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6, 'julio': 7, 'agosto': 8, 'setiembre': 9,
            'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}
REQUERIDOS = {
    'surtido_motor': [f'data/surtido_motor/outputs/{SD}/{f}' for f in ('df_matches.csv', 'df_sin_pareja.csv', 'estado_matching.json')],
    'champion_motor': [f'data/champion_motor/outputs/{CD}/{f}' for f in ('df_matches.csv', 'meta_pares_congelados.json')]
                      + ['data/champion_motor/1. Base de clientes desagregada por mes.xlsx', 'data/champion_motor/Base Minimayorista vf.xlsx'],
    'cpb': [], 'cpm': [],      # sus pares vienen de lo publicado en BigQuery
}
for p in PROGRAMAS:
    for r in REQUERIDOS.get(p, []):
        print('OK    ' if (RAIZ / r).exists() else 'FALTA ', r)
    if p == 'champion_motor':
        mes = pd.Timestamp(MES_CIERRE).month
        lista = sorted((RAIZ / 'data/champion_motor').glob('BD_Clientes_*Snack*.xlsx'))
        del_mes = [f.name for f in lista if next((m for k, m in MESES_ES.items() if k in f.name.lower()), None) == mes]
        print('OK    ' if del_mes else 'FALTA ', f'lista de clientes del mes {mes} (nombre con el mes): {del_mes or "ninguna; se usaría la más reciente: " + str([f.name for f in lista][-1:])}')
        for opc in ('equivalencias_codigos.csv', 'segmentos_manual.csv'):
            print('ok    ' if (RAIZ / 'data/champion_motor' / opc).exists() else '(sin) ', f'data/champion_motor/{opc} (opcional)')

## 2. Correr

In [ ]:
corrida = correr_mes(MES_CIERRE, PROGRAMAS, modo=MODO, raiz=RAIZ,
                     validar_cerrados=VALIDAR_CERRADOS, tolerancia_celda=TOLERANCIA_CELDA,
                     aceptar_deriva=ACEPTAR_DERIVA, motivo_deriva=MOTIVO_DERIVA,
                     reproceso=REPROCESO, reabrir_desde=REABRIR_DESDE, motivo=MOTIVO)
corrida.resumen

## 3. Revisar
- **frontera** = el mes anterior al cierre (octubre: `2026-09-01`).
- **se publican** = el mes del cierre; en surtido además los meses del G04.
- **celdas fuera de tolerancia**: esperables y explicadas: abril-Galletas G01 y julio G03 (su retroactivo se completa con octubre). Otra cosa: no cargar y revisar.

In [ ]:
for p in PROGRAMAS:
    r = corrida.res.get(p, {})
    print(f"\n══ {p}: frontera {r.get('frontera')} · se publican {r.get('incr_ton', {}).get('meses_a_publicar')} · "
          f"{r.get('incr_ton', {}).get('filas_nuevas', 0):,} filas · error: {r.get('error')}")
    for k, nombre in (('celdas_ton', 'toneladas'), ('celdas_bre', 'brechas')):
        cel = r.get(k)
        if cel is not None and len(cel):
            fuera = cel[cel['fuera'].astype(bool)]
            print(f"   {nombre}: {len(cel) - len(fuera)} de {len(cel)} celdas cerradas dentro de tolerancia")
            if len(fuera):
                print(fuera.round({'publicado': 3, 'recalculado': 3, 'dif': 3, 'dif_pct': 2}).to_string(index=False))

## 4. Vista previa (en `dry_run`): lo que quedaría publicado si se cargara
Un Excel por programa en `exportes/vista_previa_AAAAMM/` con las mismas hojas que la exportación final. No carga nada. Cada prueba del mismo mes lo reemplaza.

In [ ]:
if MODO == 'dry_run':
    previa = vista_previa(corrida, desde='2026-01-01', raiz=RAIZ)

## 5. Exportar (después de `cargar`)
Desde lo publicado: Toneladas, Brechas (par-mes y por celda), Espejos con vigencia; con fecha de reporte y estado. Archivos en `exportes/`.

In [ ]:
if MODO == 'cargar':
    salida = exportar(PROGRAMAS, desde='2026-01-01', modo='vigente', destino='excel', raiz=RAIZ)